# 🗺️ Projection Homographique sur Terrain 2D
**Modèle : RT-DETR** | Détection joueurs + ballon + projection tactique en temps réel

---
### Pipeline
1. Installation des dépendances
2. Sélection manuelle des 4 coins du terrain
3. Calcul de la matrice d'homographie
4. Détection & tracking RT-DETR
5. Projection des joueurs sur terrain 2D côte à côte

## ⚙️ 1. Installation

In [ ]:
!pip install ultralytics supervision

## 📦 2. Imports & Config

In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv

# ─── CONFIG ───────────────────────────────────────────
TERRAIN_W  = 1050          # largeur terrain 2D en pixels (105m x10)
TERRAIN_H  = 680           # hauteur terrain 2D en pixels (68m x10)
VIDEO_PATH = "test2.mp4"   # ⚠️ chemin vers ta vidéo
OUTPUT_PATH = "output_homography.mp4"
CONF       = 0.5           # seuil de confiance RT-DETR
# ──────────────────────────────────────────────────────

print("✅ Imports OK")

✅ Imports OK


## 🤖 3. Chargement du modèle RT-DETR

In [3]:
# RT-DETR-L : bon équilibre vitesse / précision
# RT-DETR-X : meilleure précision (nécessite plus de VRAM)
model   = YOLO("rtdetr-x.pt")   # téléchargement automatique ~260MB
tracker = sv.ByteTrack()

print("✅ Modèle RT-DETR chargé !")

✅ Modèle RT-DETR chargé !


## 📌 4. Sélection des 4 coins du terrain

> **Instructions :**
> 1. Une fenêtre va s'ouvrir avec la première frame de ta vidéo
> 2. **Clique sur 4 coins du terrain** dans cet ordre : ↖️ haut-gauche → ↗️ haut-droit → ↘️ bas-droit → ↙️ bas-gauche
> 3. Appuie sur **n'importe quelle touche** pour valider
>
> 💡 Conseil : utilise des **intersections de lignes** visibles (coins surface, cercle central...)

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
cap.release()

if not ret:
    raise FileNotFoundError(f"❌ Impossible d'ouvrir la vidéo : {VIDEO_PATH}")

src_points = []
frame_copy = first_frame.copy()

def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(src_points) < 4:
        src_points.append([x, y])
        cv2.circle(frame_copy, (x, y), 8, (0, 255, 0), -1)
        cv2.putText(frame_copy, str(len(src_points)), (x + 10, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
        cv2.imshow("📌 Selectionne 4 coins du terrain", frame_copy)
        if len(src_points) == 4:
            print("✅ 4 points sélectionnés ! Appuie sur une touche pour continuer.")

cv2.imshow("📌 Selectionne 4 coins du terrain", frame_copy)
cv2.setMouseCallback("📌 Selectionne 4 coins du terrain", click_event)
cv2.waitKey(0)
cv2.destroyAllWindows()

print(f"Points sélectionnés : {src_points}")

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window_w32.cpp:2552: error: (-27:Null pointer) NULL window: '📌 Selectionne 4 coins du terrain' in function 'cvSetMouseCallback'


: 

## 🔢 5. Calcul de la matrice d'homographie

In [ ]:
# Points destination : les 4 coins du terrain 2D standardisé
dst_points = np.float32([
    [0, 0],
    [TERRAIN_W, 0],
    [TERRAIN_W, TERRAIN_H],
    [0, TERRAIN_H]
])

H, status = cv2.findHomography(np.float32(src_points), dst_points)

print("✅ Matrice d'homographie calculée :")
print(np.round(H, 4))

## 🎨 6. Fonctions utilitaires (terrain + projection)

In [ ]:
def draw_pitch(w=TERRAIN_W, h=TERRAIN_H):
    """Terrain style tableau tactique — fond blanc, lignes noires."""
    pitch = np.ones((h, w, 3), dtype=np.uint8) * 245
    c, t = (30, 30, 30), 3

    cv2.rectangle(pitch, (10, 10), (w - 10, h - 10), c, t)
    cv2.line(pitch, (10, h // 2), (w - 10, h // 2), c, t)
    cv2.circle(pitch, (w // 2, h // 2), int(h * 0.14), c, t)
    cv2.circle(pitch, (w // 2, h // 2), 5, c, -1)

    # Surfaces
    cv2.rectangle(pitch, (int(w * 0.21), 10),           (int(w * 0.79), int(h * 0.157)), c, t)
    cv2.rectangle(pitch, (int(w * 0.21), int(h * 0.843)), (int(w * 0.79), h - 10),       c, t)

    # Petits rectangles
    cv2.rectangle(pitch, (int(w * 0.37), 10),           (int(w * 0.63), int(h * 0.057)), c, t)
    cv2.rectangle(pitch, (int(w * 0.37), int(h * 0.943)), (int(w * 0.63), h - 10),       c, t)

    # Arcs de cercle
    cv2.ellipse(pitch, (w // 2, int(h * 0.157)), (int(h * 0.1), int(h * 0.1)), 0,  30, 150, c, t)
    cv2.ellipse(pitch, (w // 2, int(h * 0.843)), (int(h * 0.1), int(h * 0.1)), 0, 210, 330, c, t)

    # Points penalty
    cv2.circle(pitch, (w // 2, int(h * 0.11)), 5, c, -1)
    cv2.circle(pitch, (w // 2, int(h * 0.89)), 5, c, -1)

    # Cages
    cv2.rectangle(pitch, (int(w * 0.44), 10),           (int(w * 0.56), int(h * 0.02)), c, t)
    cv2.rectangle(pitch, (int(w * 0.44), int(h * 0.98)), (int(w * 0.56), h - 10),       c, t)

    return pitch


def project_point(pt, H):
    """Projette un point (x, y) via la matrice homographique H."""
    p = np.float32([[pt]])
    proj = cv2.perspectiveTransform(p, H)
    return int(proj[0][0][0]), int(proj[0][0][1])


# Couleurs (BGR)
PLAYER_COLOR = (50,  50, 220)   # rouge
BALL_COLOR   = (0,  215, 255)   # jaune

print("✅ Fonctions prêtes")

## 🎬 7. Boucle principale — Détection + Tracking + Projection

In [ ]:
cap    = cv2.VideoCapture(VIDEO_PATH)
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps, (1360, 440)
)

box_annotator   = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator()
trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=40)

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"📹 Vidéo : {frame_count} frames à {fps:.1f} FPS")
print("   Appuie sur 'q' pour arrêter\n")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # ✅ Détection RT-DETR
    results    = model(frame, conf=CONF)[0]
    detections = sv.Detections.from_ultralytics(results)

    # ✅ Filtrer : personnes (0) et ballon (32 dans COCO)
    mask       = np.isin(detections.class_id, [0, 32])
    detections = detections[mask]
    detections = tracker.update_with_detections(detections)

    # Terrain 2D vierge
    pitch = draw_pitch()

    for box, tid, cls_id in zip(detections.xyxy, detections.tracker_id, detections.class_id):
        foot_x = int((box[0] + box[2]) / 2)
        foot_y = int(box[3])
        tx, ty = project_point((foot_x, foot_y), H)

        if 0 <= tx < TERRAIN_W and 0 <= ty < TERRAIN_H:
            if cls_id == 32:  # ballon
                cv2.circle(pitch, (tx, ty), 10, BALL_COLOR, -1)
                cv2.putText(pitch, "ball", (tx + 12, ty + 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, BALL_COLOR, 2)
            else:             # joueur
                cv2.circle(pitch, (tx, ty), 12, PLAYER_COLOR, -1)
                cv2.putText(pitch, f"#{tid}", (tx + 14, ty + 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, PLAYER_COLOR, 2)

        cv2.circle(frame, (foot_x, foot_y), 5, (0, 255, 0), -1)

    # Annotations vidéo originale
    labels = [f"#{tid}" for tid in detections.tracker_id]
    frame  = trace_annotator.annotate(frame, detections)
    frame  = box_annotator.annotate(frame, detections)
    frame  = label_annotator.annotate(frame, detections, labels)

    # Vue côte à côte
    pitch_r = cv2.resize(pitch, (680, 440))
    frame_r = cv2.resize(frame, (680, 440))
    combined = np.hstack([frame_r, pitch_r])

    out.write(combined)
    cv2.imshow("RT-DETR | Video + Terrain 2D", combined)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()
print(f"✅ Vidéo sauvegardée : {OUTPUT_PATH}")